### Dataset and Task Metadata

In [13]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="garments_worker_productivity",
    dataset_year="2020",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C51S6D",
    download_description="""
    mkdir -p local-data-warehouse/garments_worker_productivity 
    wget -P local-data-warehouse/garments_worker_productivity/ https://archive.ics.uci.edu/static/public/597/productivity+prediction+of+garment+employees.zip 
    unzip local-data-warehouse/garments_worker_productivity/productivity+prediction+of+garment+employees.zip -d local-data-warehouse/garments_worker_productivity/ 
    rm local-data-warehouse/garments_worker_productivity/productivity+prediction+of+garment+employees.zip
    """,
    # References
    academic_reference_bibtex="""@article{imran2021mining,
  title={Mining the productivity data of the garment industry},
  author={Imran, Abdullah Al and Rahim, Md Shamsur and Ahmed, Tanvir},
  journal={International Journal of Business Intelligence and Data Mining},
  volume={19},
  number={3},
  pages={319--342},
  year={2021},
  publisher={Inderscience Publishers (IEL)}
}

""",
    academic_reference_bibtex_key="imran2021mining",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Temporal"],
    curation_comments="""
    - We fix typos in data entries (e.g., "finishing " becomes "finishing").
    - We transform the date to datetime.
    - The associated paper conceptualizes the task as an interpretable ML task without consideration of realistic predictive scenarios. Therefore, we define a new predictive ML task.
    - We define the decision point in time as the start of the day.
    - We set the target variable to "actual_productivity", and use the targeted_productivity as an input feature.
    - We do not include all features directly for prediction, because some can't be expected to be available at the decision point in time: "smv", "wip", "over_time", "idle_time", "idle_men". To still keep as much information as possible, we impute the value of the previous recorded working day (by department and team). 
    - Because the timestamps are irregularly spaced, we additionally include the days passed sind the last recording per department and team.
    - Note that we try to keep feature engineering minimalistic and only with the sole purpose to prevent leaks with minimal loss of information.
    - It can be expected that feature engineering exposing the non-iid properties of the dataset will be crucial for good predictive performance.
    - We cannot fully exclude the possibility of leaks, since not enough information about the date of collection for some features is given.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="actual_productivity",
    problem_type="regression",
    objective_metric_name="rmse",
    stratify_on="",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

# NOTE: "date", "quarter", "department", "day", "team", "targeted_productivity", "actual_productivity" are safe to keep. For "incentive", "no_of_style_change", "no_of_workers" it's a guess.
lag_cols = ["smv", "wip", "over_time", "idle_time", "idle_men"]

df = pd.read_csv(dataset_mold.path / "garments_worker_productivity.csv")

df.department = df.department.str.strip(" ")
df.date = pd.to_datetime(df.date)

entity_cols = ["department", "team"]
date_col = "date"
lags = (1,)

assert not df.duplicated(["department", "team", "date"]).any(), \
    "Found duplicate rows for the same department-team-date."

df = df.sort_values(list(entity_cols) + [date_col]).reset_index(drop=True)

g = df.groupby(list(entity_cols), sort=False)

prev_date = g[date_col].shift(1)
df["days_since_prev_obs"] = (df[date_col] - prev_date).dt.days

for col in lag_cols:
    for lag in lags:
        df[f"{col}_lag_{lag}"] = g[col].shift(lag)

df = df.drop(columns=lag_cols)

as_cat_type = ["quarter", "department", "day", "team"]
df[as_cat_type] = df[as_cat_type].astype("category")

print("Loaded data shape:", df.shape)

Loaded data shape: (1197, 21)


In [3]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,date,quarter,department,day,team,targeted_productivity,incentive,no_of_style_change,no_of_workers,actual_productivity,days_since_prev_obs,smv_lag_1,wip_lag_1,over_time_lag_1,idle_time_lag_1,idle_men_lag_1
0,2015-01-01,Quarter1,finishing,Thursday,1,0.75,0,0,8.0,0.886500,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-03,Quarter1,finishing,Saturday,1,0.80,0,0,8.0,0.902917,2.0,3.94,NaN,960.0,0.0,0.0
2,2015-01-04,Quarter1,finishing,Sunday,1,0.80,0,0,8.0,0.915229,1.0,3.94,NaN,960.0,0.0,0.0
3,2015-01-05,Quarter1,finishing,Monday,1,0.80,0,0,8.0,0.961059,1.0,3.94,NaN,960.0,0.0,0.0
4,2015-01-06,Quarter1,finishing,Tuesday,1,0.80,0,0,8.0,0.936496,1.0,3.94,NaN,1920.0,0.0,0.0


## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,197
Columns: 16
Use sampling: False (sample size: 1,197)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['wip_lag_1', 'over_time_lag_1', 'smv_lag_1', 'no_of_workers', 'date', 'incentive', 'days_since_prev_obs', 'idle_time_lag_1', 'team', 'idle_men_lag_1']
Rows remaining as candidates after top-10 filter: 0 (of 1,197)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,date,quarter,department,day,team,targeted_productivity,incentive,no_of_style_change,no_of_workers,actual_productivity,days_since_prev_obs,smv_lag_1,wip_lag_1,over_time_lag_1,idle_time_lag_1,idle_men_lag_1
0,2015-01-01,Quarter1,finishing,Thursday,1,0.75,0,0,8.0,0.886500,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-03,Quarter1,finishing,Saturday,1,0.80,0,0,8.0,0.902917,2.0,3.94,NaN,960.0,0.0,0.0
2,2015-01-04,Quarter1,finishing,Sunday,1,0.80,0,0,8.0,0.915229,1.0,3.94,NaN,960.0,0.0,0.0
3,2015-01-05,Quarter1,finishing,Monday,1,0.80,0,0,8.0,0.961059,1.0,3.94,NaN,960.0,0.0,0.0
4,2015-01-06,Quarter1,finishing,Tuesday,1,0.80,0,0,8.0,0.936496,1.0,3.94,NaN,1920.0,0.0,0.0


In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,date,datetime64[ns],0.0,0.00,59.0,"2015-01-31 00:00:00, 2015-03-11 00:00:00, 2015-01-11 00:00:00, 2015-01-24 00:00:00, 2015-01-12 00:00:00, 2015-03-10 00:00:00, 2015-01-08 00:00:00, 2015-01-13 00:00:00, 2015-01-22 00:00:00, 2015-01-10 00:00:00"
1,wip_lag_1,float64,518.0,43.27,540.0,"1039.0, 1282.0, 1144.0, 1108.0, 983.0, 1086.0, 1209.0, 759.0, 749.0, 562.0"
2,days_since_prev_obs,float64,24.0,2.01,14.0,"1.0, 2.0, 3.0, 4.0, 5.0, 7.0, 6.0, 8.0, 10.0, 16.0"
3,smv_lag_1,float64,24.0,2.01,70.0,"3.94, 2.9, 22.52, 30.1, 4.15, 18.79, 4.6, 15.26, 25.9, 11.61"
4,over_time_lag_1,float64,24.0,2.01,143.0,"960.0, 1440.0, 6960.0, 6840.0, 1200.0, 1800.0, 10170.0, 0.0, 3360.0, 4080.0"
5,idle_time_lag_1,float64,24.0,2.01,12.0,"0.0, 3.5, 4.5, 5.0, 8.0, 2.0, 4.0, 6.5, 90.0, 150.0"
6,idle_men_lag_1,float64,24.0,2.01,10.0,"0.0, 30.0, 15.0, 10.0, 20.0, 35.0, 45.0, 25.0, 40.0, 37.0"
7,targeted_productivity,float64,0.0,0.00,9.0,"0.8, 0.7, 0.75, 0.65, 0.6, 0.5, 0.35, 0.4, 0.07"
8,no_of_workers,float64,0.0,0.00,61.0,"8.0, 58.0, 57.0, 59.0, 10.0, 56.5, 56.0, 34.0, 9.0, 12.0"
9,actual_productivity,float64,0.0,0.00,879.0,"0.8004, 0.9719, 0.8501, 0.7507, 0.8505, 1.0002, 0.7504, 0.8001, 0.8, 0.8001"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
team,1197.0,6.426901,3.463963,1.000000,12.000000
targeted_productivity,1197.0,0.729632,0.097891,0.070000,0.800000
incentive,1197.0,38.210526,160.182643,0.000000,3600.000000
no_of_style_change,1197.0,0.150376,0.427848,0.000000,2.000000
no_of_workers,1197.0,34.609858,22.197687,2.000000,89.000000
actual_productivity,1197.0,0.735091,0.174488,0.233705,1.120437
days_since_prev_obs,1173.0,1.384484,1.101348,1.000000,16.000000
smv_lag_1,1173.0,15.095303,10.945454,2.900000,54.560000
wip_lag_1,679.0,1195.238586,1852.668096,7.000000,23122.000000
over_time_lag_1,1173.0,4584.748508,3364.083787,0.000000,25920.000000


In [8]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column     rank                                   
date       1     2015-01-31 00:00:00     24   2.01
           2     2015-03-11 00:00:00     24   2.01
           3     2015-01-11 00:00:00     23   1.92
           4     2015-01-24 00:00:00     23   1.92
           5     2015-01-12 00:00:00     23   1.92
day        1               Wednesday    208  17.38
           2                  Sunday    203  16.96
           3                 Tuesday    201  16.79
           4                Thursday    199  16.62
           5                  Monday    199  16.62
department 1                  sweing    691  57.73
           2               finishing    506  42.27
quarter    1                Quarter1    360  30.08
           2                Quarter2    335  27.99
           3                Quarter4    248  20.72
           4                Quarter3    210  17.54
           5                Quarter5     44   3.68

In [9]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-0.807,-1.574,0.03,0.084,log,1859.1,1.067361e+15,exponential


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=10, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For Temporal non-IID data -> manual processing required
# TODO: Change once we know what to do with temporal data
splits = {
    0: {
        0: (
            df.sort_values("date").iloc[:int(df.shape[0]/3)].index.tolist(), 
            df.sort_values("date").iloc[int(df.shape[0]/3):].index.tolist()
                 ),
    }
}

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019ce7ea-6a0c-7d93-a62a-fc352d46299c
9ddb4e20146476f905d090033ad36055461b52d08f652e166defde7588929742
